In [1]:
from pyspark.sql import SparkSession

# spark = SparkSession.builder \
#     .appName("basic_app") \
#     .config("spark.driver.memory", "1g") \
#     .master("local[*]") \
#     .getOrCreate()

spark = SparkSession.builder.getOrCreate()

print(spark.version)

df = spark.createDataFrame(
    [(1, "a"), (2, "b"), (3, "c")],
    ("id", "value")
)

df.show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/22 13:12:17 WARN Utils: Your hostname, cachyos-x8664, resolves to a loopback address: 127.0.1.1; using 192.168.1.13 instead (on interface wlan0)
26/08/22 13:12:17 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/22 13:12:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


4.2.0


+---+-----+
| id|value|
+---+-----+
|  1|    a|
|  2|    b|
|  3|    c|
+---+-----+



In [2]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("date", StringType(), False),
    StructField("product", StringType(), False),
    StructField("price", DoubleType(), False),
    StructField("quantity", IntegerType(), False)
])

df = spark.read.csv("sales.csv", schema=schema, header=True)
df.show()

+---+----------+----------+------+--------+
| id|      date|   product| price|quantity|
+---+----------+----------+------+--------+
|  1|2024-01-10|    Laptop|1200.0|       1|
|  2|2024-01-11|     Mouse|  25.0|       3|
|  3|2024-01-12|  Keyboard|  45.0|       2|
|  4|2024-01-13|   Monitor| 300.0|       1|
|  5|2024-01-14|    Laptop|1250.0|       1|
|  6|2024-01-15|     Mouse|  20.0|       4|
|  7|2024-01-16|   Monitor| 290.0|       2|
|  8|2024-01-17|  Keyboard|  50.0|       1|
|  9|2024-01-18|Headphones|  80.0|       5|
| 10|2024-01-19|    Laptop|1300.0|       1|
+---+----------+----------+------+--------+



In [3]:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- date: string (nullable = true)
 |-- product: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)



In [4]:
from pyspark.sql import functions as F

df = df.withColumn("total", F.col("price") * F.col("quantity"))
df.filter(F.col("total") < 1000).show()

+---+----------+----------+-----+--------+-----+
| id|      date|   product|price|quantity|total|
+---+----------+----------+-----+--------+-----+
|  2|2024-01-11|     Mouse| 25.0|       3| 75.0|
|  3|2024-01-12|  Keyboard| 45.0|       2| 90.0|
|  4|2024-01-13|   Monitor|300.0|       1|300.0|
|  6|2024-01-15|     Mouse| 20.0|       4| 80.0|
|  7|2024-01-16|   Monitor|290.0|       2|580.0|
|  8|2024-01-17|  Keyboard| 50.0|       1| 50.0|
|  9|2024-01-18|Headphones| 80.0|       5|400.0|
+---+----------+----------+-----+--------+-----+



In [5]:
from pyspark.sql import functions as F

df = df.withColumn("date", F.to_date(F.col("date"), "yyyy-MM-dd"))
df = df.withColumn("weekday", F.date_format(F.col("date"), "EEEE"))
df = df.groupBy("weekday").agg(
    F.sum("total").alias("weekday_total"),
    F.listagg("product", " ").alias("products"),
    F.first("date")
)
df.show()

+---------+-------------+----------------+-----------+
|  weekday|weekday_total|        products|first(date)|
+---------+-------------+----------------+-----------+
|Wednesday|       1250.0| Laptop Keyboard| 2024-01-10|
|  Tuesday|        580.0|         Monitor| 2024-01-16|
|   Friday|       1390.0| Keyboard Laptop| 2024-01-12|
| Thursday|        475.0|Mouse Headphones| 2024-01-11|
| Saturday|        300.0|         Monitor| 2024-01-13|
|   Monday|         80.0|           Mouse| 2024-01-15|
|   Sunday|       1250.0|          Laptop| 2024-01-14|
+---------+-------------+----------------+-----------+



In [6]:
df_bad = spark.read.option("mode", "DROPMALFORMED").csv("sales_bad.csv", schema, ",", header=True)
df_bad.show(20)

+---+----------+----------+------+--------+
| id|      date|   product| price|quantity|
+---+----------+----------+------+--------+
|  1|2024-01-10|    Laptop|1200.0|       1|
|  2|2024-01-11|     Mouse|  25.0|       3|
|  3|2024-01-12|  Keyboard|  45.0|       2|
|  4|2024-01-13|   Monitor| 300.0|       1|
|  5|2024-01-14|    Laptop|1250.0|       1|
|  6|2024-01-15|     Mouse|  20.0|       4|
|  7|2024-01-16|   Monitor| 290.0|       2|
|  8|2024-01-17|  Keyboard|  50.0|       1|
|  9|2024-01-18|Headphones|  80.0|       5|
| 10|2024-01-19|    Laptop|1300.0|       1|
+---+----------+----------+------+--------+



In [7]:
df_bad = spark.read.option("mode", "PERMISSIVE").csv("sales_bad.csv", schema, ",", header=True)
df_bad.count()

11

In [8]:
df_bad = spark.read.option("mode", "PERMISSIVE").option("columnNameOfCorruptRecord", "_corrupt_record").csv("sales_bad.csv", schema=schema, sep=",", header=True)
df_bad.show(20)

+---+----------+----------+------+--------+
| id|      date|   product| price|quantity|
+---+----------+----------+------+--------+
|  1|2024-01-10|    Laptop|1200.0|       1|
|  2|2024-01-11|     Mouse|  25.0|       3|
|  3|2024-01-12|  Keyboard|  45.0|       2|
|  4|2024-01-13|   Monitor| 300.0|       1|
|  5|2024-01-14|    Laptop|1250.0|       1|
|  6|2024-01-15|     Mouse|  20.0|       4|
|  7|2024-01-16|   Monitor| 290.0|       2|
|  8|2024-01-17|  Keyboard|  50.0|       1|
|  9|2024-01-18|Headphones|  80.0|       5|
| 10|2024-01-19|    Laptop|1300.0|       1|
| 11|2024-01-20|  Portátil|  NULL|       2|
+---+----------+----------+------+--------+



In [9]:
from pyspark.sql import functions as F

schema_drop = StructType([
    StructField("id", IntegerType(), True),
    StructField("date", StringType(), True),
    StructField("product", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", IntegerType(), True),
])

df_drop = (spark.read
           .option("header", True)
           .option("sep", ",")
           .option("mode", "DROPMALFORMED")
           .schema(schema_drop)
           .csv("sales_bad.csv"))

print("Count con DROPMALFORMED:", df_drop.filter(F.col("price").isNotNull()).count())
df_drop.show(20)

schema_perm = StructType([
    StructField("id", IntegerType(), True),
    StructField("date", StringType(), True),
    StructField("product", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("_corrupt_record", StringType(), True)
])

df_permissive = (spark.read
                 .option("header", True)
                 .option("sep", ",")
                 .option("mode", "PERMISSIVE")
                 .option("columnNameOfCorruptRecord", "_corrupt_record")
                 .schema(schema_perm)
                 .csv("sales_bad.csv"))

print("Count con PERMISSIVE:", df_permissive.count())
df_permissive.filter(F.col("_corrupt_record").isNotNull()).show(truncate=False)
df_permissive.filter(F.col("_corrupt_record").isNull()).show(truncate=False)

Count con DROPMALFORMED: 10
+---+----------+----------+------+--------+
| id|      date|   product| price|quantity|
+---+----------+----------+------+--------+
|  1|2024-01-10|    Laptop|1200.0|       1|
|  2|2024-01-11|     Mouse|  25.0|       3|
|  3|2024-01-12|  Keyboard|  45.0|       2|
|  4|2024-01-13|   Monitor| 300.0|       1|
|  5|2024-01-14|    Laptop|1250.0|       1|
|  6|2024-01-15|     Mouse|  20.0|       4|
|  7|2024-01-16|   Monitor| 290.0|       2|
|  8|2024-01-17|  Keyboard|  50.0|       1|
|  9|2024-01-18|Headphones|  80.0|       5|
| 10|2024-01-19|    Laptop|1300.0|       1|
+---+----------+----------+------+--------+

Count con PERMISSIVE: 11
+---+----------+--------+-----+--------+-------------------------------------+
|id |date      |product |price|quantity|_corrupt_record                      |
+---+----------+--------+-----+--------+-------------------------------------+
|11 |2024-01-20|Portátil|NULL |2       |11,2024-01-20,Portátil,not_a_number,2|
+---+---------

In [10]:
from pyspark.sql import DataFrame
df_permissive.limit(5).show()

+---+----------+--------+------+--------+---------------+
| id|      date| product| price|quantity|_corrupt_record|
+---+----------+--------+------+--------+---------------+
|  1|2024-01-10|  Laptop|1200.0|       1|           NULL|
|  2|2024-01-11|   Mouse|  25.0|       3|           NULL|
|  3|2024-01-12|Keyboard|  45.0|       2|           NULL|
|  4|2024-01-13| Monitor| 300.0|       1|           NULL|
|  5|2024-01-14|  Laptop|1250.0|       1|           NULL|
+---+----------+--------+------+--------+---------------+



In [11]:
df = spark.read.csv("transactions.csv", header=True)
df.show(5)

+--------------+-----------+-------+------+-----------+----------+
|transaction_id|customer_id|country|amount|   category|      date|
+--------------+-----------+-------+------+-----------+----------+
|             1|        101|  Spain| 120.5|Electronics|2024-01-15|
|             2|        102| France|  75.0|    Fashion|2024-01-17|
|             3|        103|Germany| 210.3|       Home|2024-01-20|
|             4|        104|  Spain|  33.0|      Books|2024-02-02|
|             5|        105| France| 450.0|Electronics|2024-02-05|
+--------------+-----------+-------+------+-----------+----------+
only showing top 5 rows


In [12]:
from pyspark.sql.functions import col, year, month

df = df.withColumn("year", year(col("date")))
df = df.withColumn("month", month(col("date")))
(df.write
    .mode("overwrite")
    .partitionBy("year", "month")
    .option("compression", "snappy")
    .parquet("output/parquet/transactions"))

In [13]:
spark.read.parquet("output/parquet/transactions").filter((col("year") == 2024) & (col("month") == 1)).show()

+--------------+-----------+-------+------+-----------+----------+----+-----+
|transaction_id|customer_id|country|amount|   category|      date|year|month|
+--------------+-----------+-------+------+-----------+----------+----+-----+
|             1|        101|  Spain| 120.5|Electronics|2024-01-15|2024|    1|
|             2|        102| France|  75.0|    Fashion|2024-01-17|2024|    1|
|             3|        103|Germany| 210.3|       Home|2024-01-20|2024|    1|
+--------------+-----------+-------+------+-----------+----------+----+-----+

